# Notebook 04 — Simulation de Stabilité Temporelle

**Méthodologie corrigée** : fenêtres glissantes cumulatives sur plusieurs intervalles (15, 30, 60, 90 jours).
L'intervalle optimal de ré-entraînement est **déterminé par les données**, pas fixé a priori.

> Amélioration de notre méthodologie : l'ancienne approche (3 fenêtres fixes T1→T4) était biaisée
> car l'intervalle trimestriel avait été choisi avant la simulation.
> La nouvelle approche teste plusieurs intervalles et laisse la courbe ARI
> déterminer la fréquence optimale — pratique standard en entreprise.


In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from features import FINAL_FEATURES

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Paramètres
K            = 4      # nb clusters production
INTERVALS    = [15, 30, 60, 90]   # intervalles à tester (jours)
ARI_THRESHOLD = 0.70  # seuil de stabilité acceptable
MIN_CUSTOMERS = 100   # taille minimale d'une fenêtre valide
MIN_COMMON    = 30    # nb mini de clients communs pour ARI valide
FEATURES_SIM  = ["Log_Recency", "Frequency_flag", "Log_Monetary"]

print("Setup OK — PROJECT_ROOT:", PROJECT_ROOT)
print("Intervalles testés (jours):", INTERVALS)
print("Seuil ARI :", ARI_THRESHOLD)


Setup OK — PROJECT_ROOT: /Users/mac/Downloads/School Projects/olist-customer-segmentation
Intervalles testés (jours): [15, 30, 60, 90]
Seuil ARI : 0.7


In [17]:
# Chargement des features brutes (avec dates d'achat)
df_raw = pd.read_parquet(PROCESSED_DIR / "customer_features_raw.parquet")
df_raw["last_purchase_date"] = pd.to_datetime(df_raw["last_purchase_date"])

min_date = df_raw["last_purchase_date"].min()
max_date = df_raw["last_purchase_date"].max()
n_days   = (max_date - min_date).days

print(f"{len(df_raw):,} clients")
print(f"Plage : {min_date.date()} → {max_date.date()} ({n_days} jours)")
df_raw[["customer_unique_id", "last_purchase_date", "Recency", "Frequency", "Monetary"]].head(3)


93,358 clients
Plage : 2016-09-15 → 2018-08-29 (713 jours)


,customer_unique_id,last_purchase_date,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,111,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,114,1,27.19
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,536,1,86.22


---
## Section 1 — Méthodologie : Approche Cumulative par Fenêtres Glissantes

### Pourquoi cette approche ?

L'ancienne simulation utilisait 3 grandes fenêtres fixes (T1=mi-2017, T2=fin-2017, T3=mi-2018, T4=max).
C'est biaisé : l'intervalle trimestriel était choisi **avant** la simulation, pas découvert par elle.

### Nouvelle approche : cumulative multi-intervalles

Pour chaque intervalle W ∈ {15, 30, 60, 90} jours :
1. On définit des points de coupure espacés de W jours (après 6 mois de warm-up)
2. Pour chaque paire consécutive (t, t+W) :
   - On entraîne KMeans sur **tous** les clients connus jusqu'à t → labels₁
   - On entraîne KMeans sur **tous** les clients connus jusqu'à t+W → labels₂
   - On calcule l'ARI sur les clients communs (ceux présents avant t)
3. L'ARI moyen sur toutes les paires = stabilité pour cet intervalle

**Interprétation :** si ajouter W jours de nouvelles données ne change pas les assignments
des anciens clients (ARI ≥ 0.70), alors ré-entraîner tous les W jours est suffisant.

L'intervalle optimal = **le plus long** intervalle stable (minimise le coût de ré-entraînement).


In [ ]:
def train_on_cutoff(df: pd.DataFrame, cutoff: pd.Timestamp):
    """
    Entraîne KMeans sur tous les clients ayant acheté avant 'cutoff'.
    Recency recalculé par rapport à cutoff.
    Retourne (ids, labels) ou None si pas assez de clients.
    """
    subset = df[df["last_purchase_date"] <= cutoff].copy()
    if len(subset) < MIN_CUSTOMERS:
        return None

    subset["Log_Recency"] = np.log1p((cutoff - subset["last_purchase_date"]).dt.days)
    X = subset[FEATURES_SIM].fillna(subset[FEATURES_SIM].median()).values
    X_scaled = StandardScaler().fit_transform(X)

    km = KMeans(n_clusters=K, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    return subset["customer_unique_id"].values, labels


def compute_ari_series(df: pd.DataFrame, interval_days: int):
    """
    Calcule la série d'ARI pour un intervalle donné.
    Retourne la liste des scores ARI entre paires consécutives.
    """
    delta  = pd.Timedelta(days=interval_days)
    warmup = min_date + pd.Timedelta(days=180)

    cutoffs = []
    t = warmup
    while t <= max_date - delta:
        cutoffs.append(t)
        t += delta

    if len(cutoffs) < 2:
        return []

    scores = []
    for i in range(len(cutoffs) - 1):
        t1, t2 = cutoffs[i], cutoffs[i + 1]
        r1 = train_on_cutoff(df, t1)
        r2 = train_on_cutoff(df, t2)
        if r1 is None or r2 is None:
            continue

        ids1, lab1 = r1
        ids2, lab2 = r2
        common = np.intersect1d(ids1, ids2)
        if len(common) < MIN_COMMON:
            continue

        map1 = dict(zip(ids1, lab1))
        map2 = dict(zip(ids2, lab2))
        y1 = [map1[c] for c in common]
        y2 = [map2[c] for c in common]
        scores.append(adjusted_rand_score(y1, y2))

    return scores

print("Fonctions définies.")


Fonctions définies.


---
## Section 2 — Simulation : ARI par Intervalle

On lance la simulation pour chaque intervalle. Cela peut prendre quelques minutes
(plusieurs dizaines d'entraînements KMeans).


In [ ]:
print(f"Simulation — {len(INTERVALS)} intervalles | k={K} | seuil ARI={ARI_THRESHOLD}")
print("-" * 60)

sim_results = {}
for interval in INTERVALS:
    ari_scores = compute_ari_series(df_raw, interval)
    if ari_scores:
        mean_ari = float(np.mean(ari_scores))
        std_ari  = float(np.std(ari_scores))
        stable   = mean_ari >= ARI_THRESHOLD
    else:
        mean_ari = std_ari = float("nan")
        stable   = False

    sim_results[interval] = {
        "interval_days": interval,
        "n_pairs"      : len(ari_scores),
        "mean_ari"     : round(mean_ari, 4) if not np.isnan(mean_ari) else None,
        "std_ari"      : round(std_ari,  4) if not np.isnan(std_ari)  else None,
        "ari_scores"   : [round(float(a), 4) for a in ari_scores],
        "stable"       : stable,
    }
    status = "STABLE ✓" if stable else "INSTABLE ✗"
    n = len(ari_scores)
    m = mean_ari if not np.isnan(mean_ari) else 0
    s = std_ari  if not np.isnan(std_ari)  else 0
    print(f"  {interval:3d}j  |  {n:2d} paires  |  ARI = {m:.4f} ± {s:.4f}  [{status}]")


Simulation — 4 intervalles | k=4 | seuil ARI=0.7
------------------------------------------------------------
   15j  |  34 paires  |  ARI = 0.8197 ± 0.1019  [STABLE ✓]
   30j  |  16 paires  |  ARI = 0.7146 ± 0.0989  [STABLE ✓]
   60j  |   7 paires  |  ARI = 0.6150 ± 0.0578  [INSTABLE ✗]
   90j  |   4 paires  |  ARI = 0.5789 ± 0.0330  [INSTABLE ✗]


In [ ]:
# Tableau récapitulatif
rows = []
for interval, r in sim_results.items():
    rows.append({
        "Intervalle" : f"{interval} jours",
        "Nb paires"  : r["n_pairs"],
        "ARI moyen"  : f"{r['mean_ari']:.4f}" if r["mean_ari"] else "N/A",
        "Ecart-type" : f"{r['std_ari']:.4f}"  if r["std_ari"]  else "N/A",
        "Stable ?"   : "Oui ✓" if r["stable"] else "Non ✗",
    })

df_results = pd.DataFrame(rows)
df_results.style.applymap(
    lambda v: "background-color:#d4edda" if "Oui" in str(v) else
              "background-color:#f8d7da" if "Non" in str(v) else "",
    subset=["Stable ?"]
)


,Intervalle,Nb paires,ARI moyen,Ecart-type,Stable ?
0,15 jours,34,0.8197,0.1019,Oui ✓
1,30 jours,16,0.7146,0.0989,Oui ✓
2,60 jours,7,0.6150,0.0578,Non ✗
3,90 jours,4,0.5789,0.0330,Non ✗


---
## Section 3 — Visualisation des Résultats


In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from features import FINAL_FEATURES

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Paramètres
K            = 4
INTERVALS    = [15, 30, 60, 90]
ARI_THRESHOLD = 0.70
MIN_CUSTOMERS = 100
MIN_COMMON    = 30
FEATURES_SIM  = ["Log_Recency", "Frequency_flag", "Log_Monetary"]

print("Setup OK — PROJECT_ROOT:", PROJECT_ROOT)
print("Intervalles testés (jours):", INTERVALS)
print("Seuil ARI :", ARI_THRESHOLD)


Setup OK — PROJECT_ROOT: /Users/mac/Downloads/School Projects/olist-customer-segmentation
Intervalles testés (jours): [15, 30, 60, 90]
Seuil ARI : 0.7


---
## Section 4 — Évolution du Silhouette Score

On mesure la qualité intrinsèque des clusters à plusieurs dates de coupure.
Une tendance décroissante signale une dégradation de la structure des clusters.


In [ ]:
# Dates de coupure pour la silhouette (6 points sur la plage totale)
cutoff_dates = pd.date_range(
    start=min_date + pd.Timedelta(days=180),
    end=max_date,
    periods=6
)

silhouette_by_cutoff = {}
for cutoff in cutoff_dates:
    result = train_on_cutoff(df_raw, cutoff)
    if result is None:
        continue
    ids, labels = result
    subset = df_raw[df_raw["last_purchase_date"] <= cutoff].copy()
    subset["Log_Recency"] = np.log1p((cutoff - subset["last_purchase_date"]).dt.days)
    X = subset[FEATURES_SIM].fillna(subset[FEATURES_SIM].median()).values
    X_scaled = StandardScaler().fit_transform(X)
    try:
        sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    except Exception:
        sil = float("nan")
    silhouette_by_cutoff[cutoff.date()] = round(float(sil), 4)
    print(f"  {cutoff.date()} | {len(subset):,} clients | Silhouette = {sil:.4f}")


  2017-03-14 | 3,521 clients | Silhouette = 0.3847
  2017-06-29 | 13,269 clients | Silhouette = 0.3901
  2017-10-13 | 26,838 clients | Silhouette = 0.3904
  2018-01-28 | 47,737 clients | Silhouette = 0.3924
  2018-05-15 | 72,066 clients | Silhouette = 0.3884
  2018-08-29 | 93,358 clients | Silhouette = 0.4043


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
dates_x  = list(silhouette_by_cutoff.keys())
sil_vals = list(silhouette_by_cutoff.values())

ax.plot(dates_x, sil_vals, "o-", color="#2980b9", linewidth=2, markersize=8)
ax.axhline(0.35, color="orange", linestyle="--", linewidth=1.2,
           label="Seuil alerte (0.35)")
ax.fill_between(dates_x, sil_vals, 0.35,
                where=[v < 0.35 for v in sil_vals],
                alpha=0.2, color="red", label="Zone critique")
ax.set_xlabel("Date de coupure")
ax.set_ylabel("Silhouette Score")
ax.set_title("Evolution du Silhouette Score dans le temps")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "stability_silhouette_evolution.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 5 — Bootstrap Variance (Stabilité Interne)

30 rééchantillonnages avec remise sur la dernière fenêtre.
Mesure si le modèle est robuste à de légères variations dans les données.


In [ ]:
N_BOOTSTRAP = 30
bootstrap_sil = []
rng = np.random.default_rng(42)

df_last = df_raw.copy()
df_last["Log_Recency"] = np.log1p((max_date - df_last["last_purchase_date"]).dt.days)
X_last = df_last[FEATURES_SIM].fillna(df_last[FEATURES_SIM].median()).values
X_last_scaled = StandardScaler().fit_transform(X_last)

for i in range(N_BOOTSTRAP):
    idx     = rng.integers(0, len(X_last_scaled), size=len(X_last_scaled))
    X_boot  = X_last_scaled[idx]
    labels  = KMeans(n_clusters=K, init="k-means++", n_init=5, random_state=i).fit_predict(X_boot)
    try:
        sil = silhouette_score(X_boot, labels, sample_size=5000, random_state=42)
    except Exception:
        sil = float("nan")
    bootstrap_sil.append(float(sil))

boot_mean = float(np.nanmean(bootstrap_sil))
boot_std  = float(np.nanstd(bootstrap_sil))
boot_cv   = boot_std / boot_mean if boot_mean > 0 else float("nan")
boot_ci95 = (
    float(np.nanpercentile(bootstrap_sil, 2.5)),
    float(np.nanpercentile(bootstrap_sil, 97.5))
)

print(f"Silhouette moyen  : {boot_mean:.4f}")
print(f"Ecart-type        : {boot_std:.4f}")
print(f"CV                : {boot_cv:.4f}  ({'robuste ✓' if boot_cv < 0.05 else 'instable ⚠'})")
print(f"IC 95%            : [{boot_ci95[0]:.4f}, {boot_ci95[1]:.4f}]")


Silhouette moyen  : 0.3999
Ecart-type        : 0.0048
CV                : 0.0120  (robuste ✓)
IC 95%            : [0.3902, 0.4068]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bootstrap_sil, bins=20, color="#3498db", edgecolor="white", alpha=0.85)
ax.axvline(boot_mean, color="red",    linestyle="--", linewidth=1.5, label=f"Moyenne ({boot_mean:.4f})")
ax.axvline(boot_ci95[0], color="orange", linestyle=":", linewidth=1.2, label=f"IC 95% [{boot_ci95[0]:.4f}, {boot_ci95[1]:.4f}]")
ax.axvline(boot_ci95[1], color="orange", linestyle=":", linewidth=1.2)
ax.set_xlabel("Silhouette Score")
ax.set_ylabel("Fréquence")
ax.set_title(f"Distribution Bootstrap (n={N_BOOTSTRAP}) — CV={boot_cv:.4f}")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "stability_bootstrap.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 6 — Recommandation Fréquence de Ré-entraînement

**Logique de décision :**
- Intervalle optimal = plus long intervalle stable (ARI ≥ seuil) → minimise le coût
- Si aucun intervalle n'est stable → recommandation conservatrice (trimestriel)
- Le bootstrap valide la robustesse du modèle (CV < 0.05 = robuste)


In [ ]:
# Détermination de l'intervalle optimal
stable_intervals = [i for i, r in sim_results.items() if r["stable"]]

if stable_intervals:
    optimal_days = max(stable_intervals)  # plus long = moins coûteux
    if optimal_days <= 15:
        recommendation = "bimensuel (toutes les 2 semaines)"
    elif optimal_days <= 30:
        recommendation = "mensuel"
    elif optimal_days <= 60:
        recommendation = "bimestriel"
    else:
        recommendation = "trimestriel"
    rationale = (
        f"Le plus long intervalle stable est {optimal_days}j "
        f"(ARI={sim_results[optimal_days]['mean_ari']:.4f} >= {ARI_THRESHOLD}). "
        f"Ré-entraîner plus souvent n'apporterait pas de gain mesurable."
    )
else:
    optimal_days   = max(INTERVALS)
    recommendation = "trimestriel (conservateur)"
    rationale = "Aucun intervalle testé n'a atteint le seuil de stabilité."

# Signal bootstrap
if boot_cv < 0.05:
    boot_signal = f"Modèle robuste (CV={boot_cv:.4f} < 0.05) — ré-entraînement non urgent."
else:
    boot_signal = f"Modèle instable (CV={boot_cv:.4f} >= 0.05) — surveiller la dérive."

print("=" * 60)
print(f"RECOMMANDATION : {recommendation.upper()}")
print(f"Intervalle optimal : {optimal_days} jours")
print(f"Justification : {rationale}")
print(f"Bootstrap : {boot_signal}")
print("=" * 60)


RECOMMANDATION : MENSUEL
Intervalle optimal : 30 jours
Justification : Le plus long intervalle stable est 30j (ARI=0.7146 >= 0.7). Ré-entraîner plus souvent n'apporterait pas de gain mesurable.
Bootstrap : Modèle robuste (CV=0.0120 < 0.05) — ré-entraînement non urgent.


---
## Section 7 — Export du Rapport de Stabilité

Génère `data/processed/stability_report.json` — lu par le dashboard et le pipeline CI/CD.


In [ ]:
stability_report = {
    "generated_at"          : datetime.now().isoformat(),
    "methodology"           : "rolling_windows_corrected",
    "description"           : (
        "Simulation corrigée : approche cumulative multi-intervalles. "
        "L'intervalle optimal est déterminé par les données "
        "(plus long intervalle avec ARI moyen >= seuil)."
    ),
    "parameters"            : {
        "k"               : K,
        "features"        : FEATURES_SIM,
        "intervals_tested": INTERVALS,
        "ari_threshold"   : ARI_THRESHOLD,
        "min_customers"   : MIN_CUSTOMERS,
        "min_common"      : MIN_COMMON,
    },
    "simulation_results"    : sim_results,
    "optimal_interval_days" : optimal_days,
    "recommendation"        : recommendation,
    "rationale"             : rationale,
    "bootstrap"             : {
        "n_iterations"    : N_BOOTSTRAP,
        "mean_silhouette" : round(boot_mean, 4),
        "std_silhouette"  : round(boot_std,  4),
        "cv"              : round(boot_cv,   4),
        "ci_95"           : [round(boot_ci95[0], 4), round(boot_ci95[1], 4)],
        "signal"          : boot_signal,
    },
    "silhouette_by_cutoff"  : {str(k): v for k, v in silhouette_by_cutoff.items()},
}

import json
out = PROCESSED_DIR / "stability_report.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(stability_report, f, indent=2, ensure_ascii=False, default=str)

print(f"Rapport exporté : {out}")
print(f"Recommandation finale : {recommendation} ({optimal_days} jours)")


Rapport exporté : /Users/mac/Downloads/School Projects/olist-customer-segmentation/data/processed/stability_report.json
Recommandation finale : mensuel (30 jours)
